# Klassifikation mit allen Werten

`JobSatisfaction` als Zielwert


In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

## Daten laden

Im One-Hot-Encoded Datensatz sind die One-Hot-Encoded Spalten mit bool Werten. Um es etwas einfacher zu gestalten werden diese Werte hier in Integer-Werte (False -> 0; True -> 1) umgewandelt.

In [2]:
df = pd.read_csv("survey_results_cleaned_final.csv")

bool_cols = df.select_dtypes(include=['bool']).columns

for col in bool_cols:
    df[col] = df[col].astype(int)
df.dtypes

Unnamed: 0                          int64
ResponseId                          int64
MainBranch                         object
Age                                object
EdLevel                            object
Employment                         object
Country                            object
WorkExp                           float64
LearnCodeAI                        object
YearsCode                         float64
DevType                            object
OrgSize                            object
ICorPM                             object
RemoteWork                         object
Industry                           object
AIThreat                           object
NewRole                            object
LanguageChoice                     object
LanguageHaveWorkedWith             object
DatabaseChoice                     object
DatabaseHaveWorkedWith             object
PlatformChoice                     object
PlatformHaveWorkedWith             object
WebframeChoice                    

## Zielvariable in Klassen einteilen 

Dadurch haben wir mehr Trainingsdaten, als wenn wir `JobSatisfaction` von 0-10 als Klassen defonieren würden

Klassen
- **Low**: 0–3
- **Medium**: 4–6
- **High**: 7–10


In [3]:
# Nur Zeilen behalten, wo JobSatisfaction vorhanden ist
df = df.dropna(subset=["JobSatisfaction"]).copy()

def map_JobSatisfaction(x):
    x = float(x)
    if x <= 3:
        return "Low"
    elif x <= 6:
        return "Medium"
    else:
        return "High"

## Feature-Spalten bestimmen

- Textspalten: `object` (Strings)
- Numerische Spalten: `int/float`



In [4]:
text_cols = df.select_dtypes(include=["object"]).columns.tolist()

df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

num_cols = [c for c in num_cols if c != "JobSatisfaction"]

df = df.dropna(subset=["__text__"] + num_cols + ["JobSatisfaction"])

y = df["JobSatisfaction"].apply(map_JobSatisfaction)

print("Textspalten:", text_cols)
print("Numerische Spalten:", num_cols)

X = df[["__text__"] + num_cols].copy()

X.head()

Textspalten: ['MainBranch', 'Age', 'EdLevel', 'Employment', 'Country', 'LearnCodeAI', 'DevType', 'OrgSize', 'ICorPM', 'RemoteWork', 'Industry', 'AIThreat', 'NewRole', 'LanguageChoice', 'LanguageHaveWorkedWith', 'DatabaseChoice', 'DatabaseHaveWorkedWith', 'PlatformChoice', 'PlatformHaveWorkedWith', 'WebframeChoice', 'WebframeHaveWorkedWith', 'DevEnvsChoice', 'DevEnvsHaveWorkedWith', 'OfficeStackAsyncHaveWorkedWith', 'CommPlatformHaveWorkedWith', 'AIModelsChoice', 'AIModelsHaveWorkedWith', 'AISelect', 'AIAgents', 'AIAgent_Uses']
Numerische Spalten: ['Unnamed: 0', 'ResponseId', 'WorkExp', 'YearsCode', 'AgeNum', 'ConvertedCompTotal', 'RemoteCategoryNum']


,__text__,Unnamed: 0,ResponseId,WorkExp,YearsCode,AgeNum,ConvertedCompTotal,RemoteCategoryNum
0,i am a developer by profession 25-34 years old...,0,1,8.0,14.0,29.0,61659.84,1.00
1,i am a developer by profession 25-34 years old...,1,2,2.0,10.0,29.0,105102.00,0.75
3,i am a developer by profession 35-44 years old...,3,4,4.0,5.0,39.0,36435.36,1.00
6,i am a developer by profession 35-44 years old...,7,8,22.0,30.0,39.0,72000.00,1.00
11,i am a developer by profession 25-34 years old...,22,23,7.0,11.0,29.0,87500.00,1.00


## Train/Test Split



In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train class distribution:\n", y_train.value_counts(normalize=True))
print("Test class distribution:\n", y_test.value_counts(normalize=True))

Train size: 9437
Test size: 2360
Train class distribution:
 JobSatisfaction
High      0.720038
Medium    0.220409
Low       0.059553
Name: proportion, dtype: float64
Test class distribution:
 JobSatisfaction
High      0.719915
Medium    0.220339
Low       0.059746
Name: proportion, dtype: float64


## Preprocessing für Text und numerische Spalten




In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(), "__text__"),
        ("num", StandardScaler(), num_cols)
    ],
    remainder="drop"
)

## Pipeline definieren

- Preprocessing
- Feature-Selektion (SelectFromModel mit L1-LinearSVC)
- Klassifikator (LinearSVC)


In [7]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(LinearSVC(penalty="l1", dual=False, C=0.5))),
    ("classifier", LinearSVC())
])

## GridSearchCV



In [8]:
parameters = {
    "preprocessing__text__analyzer": ["word"],
    "preprocessing__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessing__text__max_df": [0.9],
    "preprocessing__text__min_df": [2],

    "classifier__C": [1.0, 2.0],
    "classifier__class_weight": [None, "balanced"],
}

grid = GridSearchCV(pipeline, param_grid=parameters, verbose=2, cv=3, n_jobs=-1)


## Grid Search + Beste Parameter


In [9]:
grid.fit(X_train, y_train)

print("Beste Performance:", grid.best_score_)
print("Beste Parameter:\n", grid.best_params_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


C:\Users\MoritzSchwarz\PycharmProjects\data-analytics-project\.venv\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Beste Performance: 0.7179186665817682
Beste Parameter:
 {'classifier__C': 1.0, 'classifier__class_weight': None, 'preprocessing__text__analyzer': 'word', 'preprocessing__text__max_df': 0.9, 'preprocessing__text__min_df': 2, 'preprocessing__text__ngram_range': (1, 1)}


## Evaluation auf Testdaten


In [10]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Classification Report (Test):")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=["Low", "Medium", "High"]))

Classification Report (Test):
              precision    recall  f1-score   support

        High       0.74      0.97      0.84      1699
         Low       0.33      0.01      0.01       141
      Medium       0.46      0.12      0.19       520

    accuracy                           0.73      2360
   macro avg       0.51      0.37      0.35      2360
weighted avg       0.66      0.73      0.65      2360

Confusion Matrix (rows=true, cols=pred):
[[   1   27  113]
 [   1   64  455]
 [   1   49 1649]]


In [11]:
y_pred_train = best_model.predict(X_train)

print("Classification Report (Train):")
print(classification_report(y_train, y_pred_train))

Classification Report (Train):
              precision    recall  f1-score   support

        High       0.75      0.97      0.85      6795
         Low       0.75      0.04      0.08       562
      Medium       0.48      0.14      0.21      2080

    accuracy                           0.73      9437
   macro avg       0.66      0.38      0.38      9437
weighted avg       0.69      0.73      0.66      9437

